# 📊 EDA – EEG Scenario Classification (9-Class)
**Dataset used:** `normal_task_features_Formatted.csv` (long format, used for EDA)  
**Additional datasets peeked:** Wide, ERD/ERS, Encoding Chaining  
**Output folder:** `eda_outputs/`

### 9 EEG Scenarios
| # | Scenario | Vietnamese |
|---|----------|------------|
| 1 | Lifting the left hand | Nâng tay trái |
| 2 | Lifting the right hand | Nâng tay phải |
| 3 | Lifting the left leg | Nâng chân trái |
| 4 | Lifting the right leg | Nâng chân phải |
| 5 | Opening the mouth | Há miệng |
| 6 | Nodding the head | Gật đầu |
| 7 | Shaking head | Lắc đầu |
| 8 | Desiring to drink water | Tôi muốn uống nước |
| 9 | Desiring to use the bathroom | Tôi muốn đi vệ sinh |


## 0. Setup & Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "eda_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
PALETTE9 = sns.color_palette("tab10", 9)

SCENARIO_LABELS = {
    1: "Lift Left Hand",   2: "Lift Right Hand",
    3: "Lift Left Leg",    4: "Lift Right Leg",
    5: "Open Mouth",       6: "Nod Head",
    7: "Shake Head",       8: "Want Water",
    9: "Use Bathroom",
}
print("Setup complete.")


## 1. Load All Datasets

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PATH_FMT  = "normal_task_features_Formatted.csv"
PATH_WIDE = "normal_task_features_wide.csv"
PATH_ERD  = "erd_ers_band_power_xlsx_-_ERD_ERS.csv"
PATH_ENC  = "encoding_chaining.csv"

df_fmt  = pd.read_csv(PATH_FMT)
df_wide = pd.read_csv(PATH_WIDE)
df_erd  = pd.read_csv(PATH_ERD)
df_enc  = pd.read_csv(PATH_ENC)

for name, df in [("Formatted", df_fmt), ("Wide", df_wide),
                 ("ERD/ERS",  df_erd),  ("Encoding", df_enc)]:
    print(f"{name:12s}: {df.shape[0]:>7,} rows × {df.shape[1]:>3} cols")


## 2. Dataset Summaries & Data Quality

In [ ]:
def dataset_summary(df, name):
    print(f"\n{'═'*55}")
    print(f"  {name}")
    print(f"{'═'*55}")
    print(f"  Shape      : {df.shape}")
    print(f"  NaN total  : {df.isnull().sum().sum()}")
    print(f"  Zero (num) : {(df.select_dtypes('number') == 0).sum().sum()}")
    # Extreme near-zero non-zero values (numerical noise)
    num = df.select_dtypes('number')
    extreme = ((num.abs() < 1e-15) & (num != 0)).sum().sum()
    print(f"  Extreme vals (|x|<1e-15, ≠0): {extreme}  → treated as 0 in modelling")
    print(f"  Dtypes: {dict(df.dtypes.value_counts())}")
    print(df.describe(include='all').T[['count','mean','std','min','max']].to_string())

dataset_summary(df_fmt,  "Formatted (Long)")
dataset_summary(df_wide, "Wide")
dataset_summary(df_erd,  "ERD/ERS Band Power")
dataset_summary(df_enc,  "Encoding Chaining")


## 3. Class Balance – All Datasets

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Class Balance – 9 EEG Scenarios", fontsize=14, fontweight='bold')

datasets_info = [
    (df_fmt,  'scenario_num', 'Formatted (long)',       'scenario_num → int'),
    (df_wide, 'scenario_num', 'Wide (model-ready)',      'scenario_num → int'),
    (df_erd,  'scenario',     'ERD/ERS Band Power',      'scenario string'),
    (df_enc,  'scenario_id',  'Encoding Chaining',       'scenario_id → int'),
]

for ax, (df, col, title, note) in zip(axes.flat, datasets_info):
    if col == 'scenario':
        counts = df[col].value_counts()
        counts.index = counts.index.str.extract(r'(\d+)')[0].astype(int)
        counts = counts.sort_index()
    else:
        counts = df[col].value_counts().sort_index()

    bars = ax.bar(counts.index, counts.values, color=PALETTE9,
                  edgecolor='white', linewidth=0.5)
    ax.set_title(f"{title}  (N={len(df):,})", fontsize=11, fontweight='bold')
    ax.set_xlabel("Scenario"); ax.set_ylabel("Count")
    ax.set_xticks(range(1, 10))
    ax.set_xticklabels([f"S{i}\n{SCENARIO_LABELS[i][:8]}" for i in range(1, 10)], fontsize=7)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                f'{int(bar.get_height())}', ha='center', fontsize=7)
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_class_balance.png", bbox_inches='tight')
plt.show()
print("Saved 01_class_balance.png")


## 4. MAV Feature Distribution – Wide Dataset

In [ ]:
mav_cols = [c for c in df_wide.columns if c.endswith('_mav')]
df_mav = df_wide[mav_cols + ['scenario_num']].melt(
    id_vars='scenario_num', var_name='Feature', value_name='MAV')

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_mav, x='scenario_num', y='MAV', palette=PALETTE9,
            flierprops=dict(marker='o', markersize=1.5, alpha=0.3), ax=ax)
ax.set_title("MAV Distribution per Scenario (Wide Dataset)", fontsize=12, fontweight='bold')
ax.set_xlabel("Scenario")
ax.set_ylabel("Mean Absolute Value")
ax.set_xticklabels([f"S{i}\n{SCENARIO_LABELS[i]}" for i in range(1, 10)], fontsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_wide_mav_boxplot.png", bbox_inches='tight')
plt.show()


## 5. Variance Distribution – Wide Dataset

In [ ]:
var_cols = [c for c in df_wide.columns if c.endswith('_variance')]
df_var = df_wide[var_cols + ['scenario_num']].melt(
    id_vars='scenario_num', var_name='Feature', value_name='Variance')

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_var, x='scenario_num', y='Variance', palette=PALETTE9,
            flierprops=dict(marker='o', markersize=1.5, alpha=0.3), ax=ax)
ax.set_title("Variance Distribution per Scenario (Wide Dataset)", fontsize=12, fontweight='bold')
ax.set_xlabel("Scenario"); ax.set_ylabel("Variance")
ax.set_xticklabels([f"S{i}\n{SCENARIO_LABELS[i]}" for i in range(1, 10)], fontsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_wide_variance_boxplot.png", bbox_inches='tight')
plt.show()


## 6. MAV by Channel & Subband – Formatted Dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("MAV by Subband & Task per Channel (Formatted)", fontsize=12, fontweight='bold')

for ax, ch in zip(axes, ['C3', 'C4', 'CZ']):
    sub = df_fmt[df_fmt['channel'] == ch]
    sns.violinplot(data=sub, x='subband', y='mav', hue='task',
                   ax=ax, inner='quart', split=False)
    ax.set_title(f"Channel: {ch}"); ax.set_xlabel("Subband"); ax.set_ylabel("MAV")
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_formatted_mav_channel_subband.png", bbox_inches='tight')
plt.show()


## 7. ERD/ERS% per Subband & Channel

In [ ]:
df_erd['scenario_num'] = df_erd['scenario'].str.extract(r'(\d+)').astype(int)
df_erd_clean = df_erd[df_erd['channel'].str.strip() != '']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ERD/ERS% per Subband per Scenario", fontsize=12, fontweight='bold')

for ax, sb in zip(axes, ['Mu', 'Low_Beta', 'High_Beta']):
    sub = df_erd_clean[df_erd_clean['subband'] == sb]
    sns.boxplot(data=sub, x='scenario_num', y='erd_ers_pct', palette=PALETTE9,
                flierprops=dict(marker='o', markersize=1, alpha=0.3), ax=ax)
    ax.set_title(f"Subband: {sb}"); ax.set_xlabel("Scenario"); ax.set_ylabel("ERD/ERS %")
    ax.axhline(0, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_erd_ers_per_subband.png", bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_erd_clean, x='channel', y='erd_ers_pct', hue='subband',
            palette='Set2', flierprops=dict(marker='o', markersize=1, alpha=0.3), ax=ax)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_title("ERD/ERS% by Channel & Subband", fontsize=12, fontweight='bold')
ax.set_xlabel("Channel"); ax.set_ylabel("ERD/ERS %")
ax.legend(title="Subband"); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_erd_ers_by_channel.png", bbox_inches='tight')
plt.show()


## 8. Encoding Chaining – Chain Ratio Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Chain Ratio by Feature & Task (Encoding Chaining)", fontsize=12, fontweight='bold')

for ax, feat in zip(axes.flat, df_enc['feature'].unique()):
    sub = df_enc[df_enc['feature'] == feat]
    sns.violinplot(data=sub, x='task', y='chain_ratio', palette='husl',
                   ax=ax, inner='quartile')
    ax.set_title(f"Feature: {feat}", fontsize=10)
    ax.set_xlabel("Task"); ax.set_ylabel("Chain Ratio")
    ax.tick_params(axis='x', rotation=20); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_encoding_chain_ratio.png", bbox_inches='tight')
plt.show()


## 9. Chain Ratio Heatmap – Scenario × Subband

In [ ]:
pivot = df_enc.groupby(['scenario_id', 'subband'])['chain_ratio'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Chain Ratio'})
ax.set_title("Mean Chain Ratio: Scenario × Subband", fontsize=12, fontweight='bold')
ax.set_xlabel("Subband"); ax.set_ylabel("Scenario")
ax.set_yticklabels([f"S{i} – {SCENARIO_LABELS[i]}" for i in range(1, 10)], rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/08_encoding_heatmap.png", bbox_inches='tight')
plt.show()


## 10. Feature Correlation – Wide Dataset

In [ ]:
feat_cols = [c for c in df_wide.columns if c not in ['subject_id','scenario','scenario_num']]
corr = df_wide[feat_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, square=True,
            linewidths=0.2, ax=ax, cbar_kws={'shrink': 0.5},
            xticklabels=True, yticklabels=True)
ax.set_title("Feature Correlation Matrix – Wide Dataset", fontsize=13, fontweight='bold')
plt.xticks(fontsize=5.5, rotation=90); plt.yticks(fontsize=5.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/09_wide_correlation_heatmap.png", bbox_inches='tight')
plt.show()


## 11. Subjects per Dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sample Count per Subject", fontsize=12, fontweight='bold')

for ax, (df, id_col, title, color) in zip(axes, [
    (df_wide, 'subject_id', 'Wide Dataset', 'steelblue'),
    (df_enc,  'subject_id', 'Encoding Chaining', 'darkorange'),
]):
    counts = df.groupby(id_col).size().reset_index(name='count')
    ax.bar(range(len(counts)), counts['count'], color=color, alpha=0.8)
    ax.set_title(f"{title} (n_subjects={len(counts)})")
    ax.set_xlabel("Subject Index"); ax.set_ylabel("Row Count")
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/10_subject_distribution.png", bbox_inches='tight')
plt.show()


## 12. Scenario Stats Summary Table

In [ ]:
stats = df_fmt.groupby('scenario_num')[['mav','variance']].agg(['mean','std','min','max'])
stats.columns = ['MAV_mean','MAV_std','MAV_min','MAV_max',
                 'Var_mean','Var_std','Var_min','Var_max']
stats.index = [f"S{i} – {SCENARIO_LABELS[i]}" for i in stats.index]
print("Scenario-wise feature statistics:")
print(stats.round(6).to_string())

# Save
stats.to_csv(f"{OUTPUT_DIR}/eda_scenario_stats.csv")

# Summary table image
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')
rows = [[idx] + [f"{v:.6f}" for v in row] for idx, row in stats.iterrows()]
t = ax.table(cellText=rows, colLabels=['Scenario'] + list(stats.columns),
             cellLoc='center', loc='center', colColours=['#4472C4']*(len(stats.columns)+1))
t.auto_set_font_size(False); t.set_fontsize(7.5); t.scale(1.2, 1.6)
for j in range(len(stats.columns)+1): t[0,j].set_text_props(color='white', fontweight='bold')
ax.set_title("Scenario Feature Statistics (Formatted)", fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/11_scenario_stats_table.png", bbox_inches='tight')
plt.show()
print("\n✅ EDA Complete – all figures saved to:", OUTPUT_DIR)
